# StruSel: Structurome-wide Selectivity of pathogen-exclusive targets

**Version:** v0.1-alpha  
**Use case:** *Klebsiella pneumoniae* vs Human structurome  
**Author:** Pranavathiyani G, SASTRA Deemed University  
**GitHub:** https://github.com/pranavathiyani/strusel  

> *Note: Claude (Anthropic) was used for code generation assistance.*

---

## Pipeline
```
Cell 0  : GPU check
Cell 1  : Install dependencies
Cell 2  : Config + directories + checkpoint system
Cell 3  : Download + extract + pLDDT filter K. pneumoniae AF2
Cell 4  : Download + extract + pLDDT filter Human AF2
Cell 5  : MMseqs2 sequence search
Cell 6  : FoldSeek GPU structural search
Cell 7  : ESM2 pLM embeddings (prototype)
Cell 8  : Four-category classification
Cell 9  : DEG essential gene filter
Cell 10 : CARD RGI AMR mapping
Cell 11 : Final tiers + PDF report
Cell 12 : UniProt GO/pathway validation
```

In [ ]:
# Cell 0: GPU + Environment Check
import torch

!nvidia-smi
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
!df -h /content
print('Cell 0 done.')

In [ ]:
# Cell 1: Install all dependencies

# FoldSeek (CUDA-enabled)
!wget -q https://mmseqs.com/foldseek/foldseek-linux-gpu.tar.gz
!tar -xzf foldseek-linux-gpu.tar.gz

# MMseqs2
!wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz
!tar -xzf mmseqs-linux-avx2.tar.gz

# DIAMOND (for RGI)
!apt-get install -q -y diamond-aligner

# Python packages
!pip install -q biopython requests pandas numpy tqdm matplotlib seaborn
!pip install -q fair-esm
!pip install -q git+https://github.com/arpcard/rgi.git

# Verify
!echo 'FoldSeek:'; ./foldseek/bin/foldseek version
!echo 'MMseqs2:';  ./mmseqs/bin/mmseqs version
!echo 'DIAMOND:';  diamond version

import torch, esm, Bio, pandas, matplotlib
print(f'PyTorch:    {torch.__version__}')
print(f'ESM:        {esm.__version__}')
print(f'Biopython:  {Bio.__version__}')
print('Cell 1 done.')

In [ ]:
# Cell 2: Config + Directories + Checkpoint system

import json
from pathlib import Path

# CONFIGURABLE PARAMETERS
PATHOGEN_NAME   = 'Klebsiella pneumoniae'
PATHOGEN_TAXID  = '1125630'
PATHOGEN_UP     = 'UP000007841'
PATHOGEN_STRAIN = 'KLEPH'
AF_VERSION      = 'v6'

KPNE_AF_URL  = f'https://ftp.ebi.ac.uk/pub/databases/alphafold/{AF_VERSION}/{PATHOGEN_UP}_{PATHOGEN_TAXID}_{PATHOGEN_STRAIN}_{AF_VERSION}.tar'
HUMAN_AF_URL = 'https://ftp.ebi.ac.uk/pub/databases/alphafold/v6/UP000005640_9606_HUMAN_v6.tar'
DEG_URL      = 'http://tubic.org/deg/public/download/DEG10.aa.gz'

PLDDT_CUTOFF    = 70.0
MMSEQS_EVALUE   = 1e-5
MMSEQS_SENS     = 7.5
QCOV_CUTOFF     = 0.7
TCOV_CUTOFF     = 0.7
FS_EVALUE       = 0.001
TM_HOMOLOG      = 0.5
TM_UNRELATED    = 0.3
ESM2_SIM_CUTOFF = 0.5

FOLDSEEK_BIN = '/content/foldseek/bin/foldseek'
MMSEQS_BIN   = '/content/mmseqs/bin/mmseqs'

BASE_DIR        = Path('/content/strusel')
KPNE_FILTERED   = BASE_DIR / 'structures' / 'kpne_filtered'
HUMAN_FILTERED  = BASE_DIR / 'structures' / 'human_filtered'
FASTA_DIR       = BASE_DIR / 'fasta'
MMSEQS_DIR      = BASE_DIR / 'mmseqs'
FOLDSEEK_DIR    = BASE_DIR / 'foldseek_run'
ESM2_DIR        = BASE_DIR / 'esm2'
DEG_DIR         = BASE_DIR / 'deg'
RESULTS_DIR     = BASE_DIR / 'results'
CHECKPOINT_FILE = BASE_DIR / 'checkpoint.json'

for d in [KPNE_FILTERED, HUMAN_FILTERED, FASTA_DIR, MMSEQS_DIR,
          FOLDSEEK_DIR, ESM2_DIR, DEG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def save_checkpoint(data):
    existing = load_checkpoint()
    existing.update(data)
    json.dump(existing, open(CHECKPOINT_FILE, 'w'), indent=2)
    print(f'  [checkpoint saved: {list(data.keys())}]')

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        return json.load(open(CHECKPOINT_FILE))
    return {}

def checkpoint_exists(key):
    return key in load_checkpoint()

print(f'Pathogen      : {PATHOGEN_NAME} (TaxID: {PATHOGEN_TAXID})')
print(f'pLDDT cutoff  : {PLDDT_CUTOFF}')
print(f'TM homolog    : >= {TM_HOMOLOG}')
print(f'TM unrelated  : < {TM_UNRELATED}')
print(f'qcov/tcov     : >= {QCOV_CUTOFF}')
print('Cell 2 done.')

In [ ]:
# Cell 3: Download + Extract + pLDDT filter K. pneumoniae AF2
# Uses multiprocessing for speed

import tarfile, gzip, json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

def process_member(args):
    name, data, is_gz = args
    try:
        pdb_text = gzip.decompress(data).decode('utf-8', errors='ignore') if is_gz \
                   else data.decode('utf-8', errors='ignore')
        bfactors = []
        for line in pdb_text.splitlines():
            if line.startswith('ATOM'):
                try: bfactors.append(float(line[60:66].strip()))
                except: continue
        mean_plddt = sum(bfactors)/len(bfactors) if bfactors else 0.0
        fname     = Path(name).name.replace('.gz','')
        accession = fname.split('-')[1] if '-' in fname else fname.replace('.pdb','')
        return accession, mean_plddt, pdb_text, fname
    except:
        return None, 0.0, None, None

if checkpoint_exists('kpne_plddt_kept'):
    plddt_kept_kpne = set(load_checkpoint()['kpne_plddt_kept'])
    print(f'Checkpoint found. pLDDT kept: {len(plddt_kept_kpne)}')
else:
    KPNE_TAR = BASE_DIR / 'kpne.tar'
    print('Downloading K. pneumoniae AF2 structures...')
    !wget -q --show-progress -O {KPNE_TAR} {KPNE_AF_URL}
    print('Download complete.')
    print(f'Extracting + filtering pLDDT >= {PLDDT_CUTOFF} [{cpu_count()} cores]...')
    plddt_log = []; kept = rejected = 0
    with tarfile.open(KPNE_TAR, 'r') as tar:
        members = [m for m in tar.getmembers() if m.name.endswith('.pdb.gz') or m.name.endswith('.pdb')]
        print(f'Total structures: {len(members)}')
        args_list = []
        for m in members:
            f = tar.extractfile(m)
            if f: args_list.append((m.name, f.read(), m.name.endswith('.gz')))
    with Pool(cpu_count()) as pool:
        results = list(tqdm(pool.imap(process_member, args_list), total=len(args_list)))
    for accession, mean_plddt, pdb_text, fname in results:
        if accession is None: continue
        keep = mean_plddt >= PLDDT_CUTOFF
        plddt_log.append({'accession': accession, 'mean_plddt': round(mean_plddt,2), 'kept': keep})
        if keep and pdb_text: (KPNE_FILTERED / fname).write_text(pdb_text); kept += 1
        else: rejected += 1
    plddt_df = pd.DataFrame(plddt_log)
    plddt_df.to_csv(RESULTS_DIR / 'kpne_plddt_filter.tsv', sep='\t', index=False)
    plddt_kept_kpne = set(plddt_df[plddt_df['kept']]['accession'].tolist())
    KPNE_TAR.unlink()
    save_checkpoint({'kpne_plddt_kept': list(plddt_kept_kpne)})
    print(f'Kept: {kept} | Rejected: {rejected}')
print('Cell 3 done.')

In [ ]:
# Cell 4: Download + Extract + pLDDT filter Human AF2
# Streaming approach to avoid RAM crash on 23k structures

import tarfile, gzip
import pandas as pd
from pathlib import Path
from tqdm import tqdm

if checkpoint_exists('human_plddt_kept'):
    plddt_kept_human = set(load_checkpoint()['human_plddt_kept'])
    print(f'Checkpoint found. pLDDT kept: {len(plddt_kept_human)}')
else:
    HUMAN_TAR = BASE_DIR / 'human.tar'
    if not HUMAN_TAR.exists():
        print('Downloading Human AF2 structures (~5GB)...')
        !wget -q --show-progress -O {HUMAN_TAR} {HUMAN_AF_URL}
    def extract_mean_plddt(pdb_text):
        bfactors = []
        for line in pdb_text.splitlines():
            if line.startswith('ATOM'):
                try: bfactors.append(float(line[60:66].strip()))
                except: continue
        return sum(bfactors)/len(bfactors) if bfactors else 0.0
    print(f'Extracting + filtering pLDDT >= {PLDDT_CUTOFF} [streaming]...')
    plddt_log = []; kept = rejected = 0
    with tarfile.open(HUMAN_TAR, 'r') as tar:
        members = [m for m in tar.getmembers() if m.name.endswith('.pdb.gz') or m.name.endswith('.pdb')]
        print(f'Total structures: {len(members)}')
        for m in tqdm(members):
            try:
                f = tar.extractfile(m)
                if f is None: continue
                raw = f.read()
                pdb_text = gzip.decompress(raw).decode('utf-8', errors='ignore') if m.name.endswith('.gz') \
                           else raw.decode('utf-8', errors='ignore')
                del raw
                mean_plddt = extract_mean_plddt(pdb_text)
                fname     = Path(m.name).name.replace('.gz','')
                accession = fname.split('-')[1] if '-' in fname else fname.replace('.pdb','')
                plddt_log.append({'accession': accession, 'mean_plddt': round(mean_plddt,2), 'kept': mean_plddt >= PLDDT_CUTOFF})
                if mean_plddt >= PLDDT_CUTOFF: (HUMAN_FILTERED / fname).write_text(pdb_text); kept += 1
                else: rejected += 1
                del pdb_text
            except: continue
    plddt_df_human = pd.DataFrame(plddt_log)
    plddt_df_human.to_csv(RESULTS_DIR / 'human_plddt_filter.tsv', sep='\t', index=False)
    plddt_kept_human = set(plddt_df_human[plddt_df_human['kept']]['accession'].tolist())
    HUMAN_TAR.unlink()
    !df -h /content
    save_checkpoint({'human_plddt_kept': list(plddt_kept_human)})
    print(f'Kept: {kept} | Rejected: {rejected}')
print('Cell 4 done.')

In [ ]:
# Cell 5: MMseqs2 sequence search (K. pneumoniae vs Human)
# NOTE: MMseqs2 easy-search does NOT support --gpu flag

import pandas as pd

KPNE_FASTA  = str(FASTA_DIR / 'kpne.fasta')
HUMAN_FASTA = str(FASTA_DIR / 'human.fasta')
KPNE_DB     = str(MMSEQS_DIR / 'kpne_db')
HUMAN_DB    = str(MMSEQS_DIR / 'human_db')
MMSEQS_OUT  = str(MMSEQS_DIR / 'kpne_vs_human.tsv')
MMSEQS_TMP  = str(MMSEQS_DIR / 'tmp')

def extract_accession(header):
    parts = header.split('|')
    return parts[1] if len(parts) >= 2 else header.split()[0].replace('>','')

if checkpoint_exists('seq_nonhomologs'):
    cp = load_checkpoint()
    seq_nonhomologs = set(cp['seq_nonhomologs'])
    all_kpne_acc    = set(cp['all_kpne_acc'])
    print(f'Checkpoint found. Sequence non-homologs: {len(seq_nonhomologs)}')
else:
    if not Path(KPNE_FASTA).exists():
        !wget -q --show-progress -O {KPNE_FASTA} "https://rest.uniprot.org/uniprotkb/stream?query=organism_id:{PATHOGEN_TAXID}&format=fasta&compressed=false"
    if not Path(HUMAN_FASTA).exists():
        !wget -q --show-progress -O {HUMAN_FASTA} "https://rest.uniprot.org/uniprotkb/stream?query=organism_id:9606&format=fasta&compressed=false"
    kpne_count  = !grep -c '^>' {KPNE_FASTA}
    human_count = !grep -c '^>' {HUMAN_FASTA}
    print(f'K. pneumoniae: {kpne_count[0]} | Human: {human_count[0]}')
    !{MMSEQS_BIN} createdb {KPNE_FASTA}  {KPNE_DB}  --dbtype 1 -v 0
    !{MMSEQS_BIN} createdb {HUMAN_FASTA} {HUMAN_DB} --dbtype 1 -v 0
    print('Running MMseqs2 search...')
    !{MMSEQS_BIN} easy-search {KPNE_FASTA} {HUMAN_FASTA} {MMSEQS_OUT} {MMSEQS_TMP} \
        --format-output 'query,target,pident,alnlen,qcov,tcov,evalue,bits' \
        -e {MMSEQS_EVALUE} -s {MMSEQS_SENS} --max-seqs 1 --threads 4 -v 0
    mmseqs_df = pd.read_csv(MMSEQS_OUT, sep='\t', names=['query','target','pident','alnlen','qcov','tcov','evalue','bits'])
    mmseqs_filt = mmseqs_df[(mmseqs_df['qcov']>=QCOV_CUTOFF)&(mmseqs_df['tcov']>=TCOV_CUTOFF)]
    mmseqs_df.to_csv(RESULTS_DIR/'mmseqs_hits_raw.tsv',       sep='\t', index=False)
    mmseqs_filt.to_csv(RESULTS_DIR/'mmseqs_hits_filtered.tsv', sep='\t', index=False)
    all_kpne_acc = set()
    with open(KPNE_FASTA) as f:
        for line in f:
            if line.startswith('>'): all_kpne_acc.add(extract_accession(line.strip()))
    plddt_kept_kpne = set(load_checkpoint()['kpne_plddt_kept'])
    seq_hits             = set(mmseqs_filt['query'].apply(extract_accession).unique())
    seq_nonhomologs      = all_kpne_acc - seq_hits
    seq_nonhomologs_filt = seq_nonhomologs & plddt_kept_kpne
    print(f'Total K. pneumoniae     : {len(all_kpne_acc)}')
    print(f'Raw hits                : {len(mmseqs_df)}')
    print(f'After qcov/tcov filter  : {len(mmseqs_filt)}')
    print(f'Sequence non-homologs   : {len(seq_nonhomologs_filt)}')
    save_checkpoint({'seq_nonhomologs':list(seq_nonhomologs_filt),'seq_hits':list(seq_hits),'all_kpne_acc':list(all_kpne_acc)})
print('Cell 5 done.')

In [ ]:
# Cell 6: FoldSeek GPU structural search (K. pneumoniae vs Human AF2)
# Requires: foldseek-linux-gpu binary

import pandas as pd
from pathlib import Path

KPNE_STRUCT_DB  = str(FOLDSEEK_DIR / 'kpne_struct_db')
HUMAN_STRUCT_DB = str(FOLDSEEK_DIR / 'human_struct_db')
FS_TMP          = str(FOLDSEEK_DIR / 'tmp')
FS_OUT          = str(FOLDSEEK_DIR / 'kpne_vs_human_fs.tsv')

if checkpoint_exists('struct_nonhomologs'):
    cp = load_checkpoint()
    struct_nonhomologs = set(cp['struct_nonhomologs'])
    struct_homologs    = set(cp['struct_homologs'])
    struct_mimics      = set(cp['struct_mimics'])
    print(f'Checkpoint found.')
    print(f'Structural homologs     : {len(struct_homologs)}')
    print(f'Structural mimics       : {len(struct_mimics)}')
    print(f'Structural non-homologs : {len(struct_nonhomologs)}')
else:
    print('Building FoldSeek DBs...')
    !{FOLDSEEK_BIN} createdb {str(KPNE_FILTERED)}  {KPNE_STRUCT_DB}  --threads 4 -v 0
    !{FOLDSEEK_BIN} createdb {str(HUMAN_FILTERED)} {HUMAN_STRUCT_DB} --threads 4 -v 0
    print('Running FoldSeek search (GPU)...')
    !{FOLDSEEK_BIN} easy-search {str(KPNE_FILTERED)} {HUMAN_STRUCT_DB} {FS_OUT} {FS_TMP} \
        --format-output 'query,target,evalue,bits,alntmscore,qtmscore,ttmscore,lddt,lddtfull,alnlen,qlen,tlen,qcov,tcov' \
        --exhaustive-search 0 -e {FS_EVALUE} --gpu 1 --threads 4 -v 1
    fs_cols = ['query','target','evalue','bits','alntmscore','qtmscore','ttmscore','lddt','lddtfull','alnlen','qlen','tlen','qcov','tcov']
    fs_df   = pd.read_csv(FS_OUT, sep='\t', names=fs_cols)
    fs_filt = fs_df[(fs_df['qcov']>=QCOV_CUTOFF)&(fs_df['tcov']>=TCOV_CUTOFF)].copy()
    def acc_from_fname(fname):
        fname = Path(fname).name.replace('.pdb','')
        parts = fname.split('-')
        return parts[1] if len(parts) >= 2 else fname
    fs_filt['query_acc']  = fs_filt['query'].apply(acc_from_fname)
    fs_filt['target_acc'] = fs_filt['target'].apply(acc_from_fname)
    best_hits = fs_filt.loc[fs_filt.groupby('query_acc')['qtmscore'].idxmax()]
    plddt_kept_kpne    = set(load_checkpoint()['kpne_plddt_kept'])
    struct_homologs    = set(best_hits[best_hits['qtmscore']>=TM_HOMOLOG]['query_acc'])
    struct_mimics      = set(best_hits[(best_hits['qtmscore']>=TM_UNRELATED)&(best_hits['qtmscore']<TM_HOMOLOG)]['query_acc'])
    struct_hits_any    = set(best_hits['query_acc'])
    no_struct_hit      = plddt_kept_kpne - struct_hits_any
    low_tm             = set(best_hits[best_hits['qtmscore']<TM_UNRELATED]['query_acc'])
    struct_nonhomologs = (no_struct_hit | low_tm) & plddt_kept_kpne
    fs_df.to_csv(RESULTS_DIR/'foldseek_hits_raw.tsv',        sep='\t', index=False)
    fs_filt.to_csv(RESULTS_DIR/'foldseek_hits_filtered.tsv', sep='\t', index=False)
    best_hits.to_csv(RESULTS_DIR/'foldseek_best_hits.tsv',   sep='\t', index=False)
    print(f'Raw hits                       : {len(fs_df)}')
    print(f'After qcov/tcov                : {len(fs_filt)}')
    print(f'Structural homologs  (TM>=0.5) : {len(struct_homologs)}')
    print(f'Structural mimics (0.3<=TM<0.5): {len(struct_mimics)}')
    print(f'Structural non-homologs (TM<0.3): {len(struct_nonhomologs)}')
    save_checkpoint({'struct_nonhomologs':list(struct_nonhomologs),'struct_homologs':list(struct_homologs),'struct_mimics':list(struct_mimics)})
print('Cell 6 done.')

In [ ]:
# Cell 7: ESM2 pLM embeddings + cosine similarity
# PROTOTYPE: targeted human subset only (hit proteins from MMseqs2 + FoldSeek)
# TODO v2: Full human proteome comparison

import torch, esm
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

if checkpoint_exists('esm2_nonhomologs'):
    esm2_nonhomologs = set(load_checkpoint()['esm2_nonhomologs'])
    print(f'Checkpoint found. ESM2 non-homologs: {len(esm2_nonhomologs)}')
else:
    mmseqs_df = pd.read_csv(RESULTS_DIR/'mmseqs_hits_filtered.tsv', sep='\t')
    fs_df     = pd.read_csv(RESULTS_DIR/'foldseek_hits_filtered.tsv', sep='\t')
    def acc_from_header(h):
        parts = str(h).split('|')
        return parts[1] if len(parts) >= 2 else str(h).split()[0]
    def acc_from_fname(f):
        f = Path(str(f)).name.replace('.pdb','')
        parts = f.split('-')
        return parts[1] if len(parts) >= 2 else f
    relevant_human = set(mmseqs_df['target'].apply(acc_from_header)) | set(fs_df['target'].apply(acc_from_fname))
    print(f'Relevant human proteins: {len(relevant_human)}')
    print('Loading ESM2 8M model (fp16)...')
    model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    model = model.half().to(DEVICE).eval()
    batch_converter = alphabet.get_batch_converter()
    def parse_fasta(fasta_path, filter_ids=None, max_len=1022):
        seqs, current_id, current_seq = {}, None, []
        with open(fasta_path) as f:
            for line in f:
                line = line.strip()
                if line.startswith('>'):
                    if current_id:
                        seq = ''.join(current_seq)
                        if len(seq) <= max_len and (filter_ids is None or current_id in filter_ids):
                            seqs[current_id] = seq
                    parts = line.split('|')
                    current_id = parts[1] if len(parts) >= 2 else line[1:].split()[0]
                    current_seq = []
                else:
                    current_seq.append(line)
            if current_id:
                seq = ''.join(current_seq)
                if len(seq) <= max_len and (filter_ids is None or current_id in filter_ids):
                    seqs[current_id] = seq
        return seqs
    def get_embeddings(seqs_dict, batch_size=64, repr_layer=6):
        ids = list(seqs_dict.keys()); seqs = list(seqs_dict.values()); embs = {}
        for i in tqdm(range(0, len(ids), batch_size)):
            b_ids = ids[i:i+batch_size]; b_seqs = seqs[i:i+batch_size]
            try:
                _, _, tokens = batch_converter(list(zip(b_ids, b_seqs)))
                tokens = tokens.to(DEVICE)
                with torch.no_grad():
                    out = model(tokens, repr_layers=[repr_layer], return_contacts=False)
                reps = out['representations'][repr_layer]
                for j, bid in enumerate(b_ids):
                    embs[bid] = reps[j, 1:len(b_seqs[j])+1].mean(0).float().cpu().numpy()
            except Exception as e: print(f'Batch {i} error: {e}')
            torch.cuda.empty_cache()
        return embs
    kpne_seqs  = parse_fasta(str(FASTA_DIR/'kpne.fasta'))
    human_seqs = parse_fasta(str(FASTA_DIR/'human.fasta'), filter_ids=relevant_human)
    print(f'K. pneumoniae: {len(kpne_seqs)} | Human (targeted): {len(human_seqs)}')
    print('Computing K. pneumoniae embeddings...')
    kpne_embs  = get_embeddings(kpne_seqs,  batch_size=64)
    print('Computing Human embeddings...')
    human_embs = get_embeddings(human_seqs, batch_size=64)
    kpne_ids  = list(kpne_embs.keys()); human_ids = list(human_embs.keys())
    kpne_mat  = torch.nn.functional.normalize(torch.tensor(np.stack([kpne_embs[k]  for k in kpne_ids]),  dtype=torch.float32).to(DEVICE), dim=1)
    human_mat = torch.nn.functional.normalize(torch.tensor(np.stack([human_embs[k] for k in human_ids]), dtype=torch.float32).to(DEVICE), dim=1)
    results_esm = []
    for i in tqdm(range(0, len(kpne_ids), 256)):
        batch = kpne_mat[i:i+256]
        sims  = torch.mm(batch, human_mat.T)
        max_sims, max_idx = sims.max(dim=1)
        for j,(sim,idx) in enumerate(zip(max_sims.cpu().numpy(), max_idx.cpu().numpy())):
            results_esm.append({'kpne_acc':kpne_ids[i+j],'human_acc':human_ids[int(idx)],'max_cos_sim':round(float(sim),4)})
        torch.cuda.empty_cache()
    esm_df = pd.DataFrame(results_esm)
    esm_df.to_csv(RESULTS_DIR/'esm2_similarities.tsv', sep='\t', index=False)
    plddt_kept_kpne  = set(load_checkpoint()['kpne_plddt_kept'])
    esm2_nonhomologs = set(esm_df[esm_df['max_cos_sim']<=ESM2_SIM_CUTOFF]['kpne_acc']) & plddt_kept_kpne
    print(f'ESM2 non-homologs (cos<={ESM2_SIM_CUTOFF}): {len(esm2_nonhomologs)}')
    print('NOTE: Prototype — targeted human subset only. TODO v2: full human comparison.')
    save_checkpoint({'esm2_nonhomologs':list(esm2_nonhomologs)})
print('Cell 7 done.')

In [ ]:
# Cell 8: Four-category classification (A/B/B2/C/C2/D/D*)

import pandas as pd
from pathlib import Path

cp = load_checkpoint()
all_kpne_acc       = set(cp['all_kpne_acc'])
plddt_kept_kpne    = set(cp['kpne_plddt_kept'])
seq_hits           = set(cp['seq_hits'])
struct_homologs    = set(cp['struct_homologs'])
struct_mimics      = set(cp['struct_mimics'])
struct_nonhomologs = set(cp['struct_nonhomologs'])

esm_df    = pd.read_csv(RESULTS_DIR/'esm2_similarities.tsv',    sep='\t')
fs_best   = pd.read_csv(RESULTS_DIR/'foldseek_best_hits.tsv',   sep='\t')
mmseqs_df = pd.read_csv(RESULTS_DIR/'mmseqs_hits_filtered.tsv', sep='\t')

def acc_from_fname(f):
    f = Path(str(f)).name.replace('.pdb','')
    parts = f.split('-')
    return parts[1] if len(parts) >= 2 else f
def acc_from_header(h):
    parts = str(h).split('|')
    return parts[1] if len(parts) >= 2 else str(h).split()[0]

fs_best['query_acc']   = fs_best['query'].apply(acc_from_fname)
mmseqs_df['query_acc'] = mmseqs_df['query'].apply(acc_from_header)
tm_scores  = dict(zip(fs_best['query_acc'],   fs_best['qtmscore']))
seq_pident = dict(zip(mmseqs_df['query_acc'], mmseqs_df['pident']))
esm_sim    = dict(zip(esm_df['kpne_acc'],     esm_df['max_cos_sim']))

records = []
for acc in plddt_kept_kpne:
    has_seq_hit = acc in seq_hits
    tm          = tm_scores.get(acc, 0.0)
    if   has_seq_hit and tm >= TM_HOMOLOG:                    category, label = 'A',  'Homolog'
    elif has_seq_hit and tm < TM_UNRELATED:                   category, label = 'B',  'Seq-similar Struct-divergent'
    elif has_seq_hit and TM_UNRELATED <= tm < TM_HOMOLOG:     category, label = 'B2', 'Seq-similar Struct-partial'
    elif not has_seq_hit and tm >= TM_HOMOLOG:                category, label = 'C',  'Structural mimic'
    elif not has_seq_hit and TM_UNRELATED <= tm < TM_HOMOLOG: category, label = 'C2', 'Partial structural mimic'
    elif not has_seq_hit and 0 < tm < TM_UNRELATED:           category, label = 'D',  'StruSel candidate'
    else:                                                     category, label = 'D*', 'High-confidence StruSel candidate'
    records.append({'accession':acc,'category':category,'label':label,'seq_hit':has_seq_hit,
                    'pident':seq_pident.get(acc,None),'tm_score':round(tm,4),
                    'struct_hit':acc in struct_homologs or acc in struct_mimics,
                    'is_mimic':acc in struct_mimics,'esm2_cos':esm_sim.get(acc,None)})

classif_df = pd.DataFrame(records)
classif_df.to_csv(RESULTS_DIR/'strusel_classification.tsv', sep='\t', index=False)

print('── StruSel Classification Summary ──────────────────────')
for _, row in classif_df.groupby(['category','label']).size().reset_index(name='count').iterrows():
    print(f"  {row['category']:4s} | {row['label']:40s} | {row['count']:5d}")
print(f"  {'':4s}   {'Total':40s}   {len(classif_df):5d}")
save_checkpoint({'classification_done': True})
print('Cell 8 done.')

In [ ]:
# Cell 9: DEG essential gene filter + cross-reference

import gzip, pandas as pd

DEG_FASTA = str(DEG_DIR/'deg_prokaryote.fasta')
DEG_DB    = str(DEG_DIR/'deg_db')
DEG_OUT   = str(DEG_DIR/'kpne_vs_deg.tsv')
DEG_TMP   = str(DEG_DIR/'tmp')

if checkpoint_exists('essential_accs'):
    essential_accs = set(load_checkpoint()['essential_accs'])
    print(f'Checkpoint found. Essential: {len(essential_accs)}')
else:
    print('Downloading DEG prokaryote AA sequences...')
    !wget -q --show-progress -O {DEG_FASTA}.gz {DEG_URL}
    with gzip.open(f'{DEG_FASTA}.gz','rb') as fi, open(DEG_FASTA,'wb') as fo: fo.write(fi.read())
    deg_count = !grep -c '^>' {DEG_FASTA}
    print(f'DEG sequences: {deg_count[0]}')
    !{MMSEQS_BIN} createdb {DEG_FASTA} {DEG_DB} --dbtype 1 -v 0
    !{MMSEQS_BIN} easy-search {str(FASTA_DIR/'kpne.fasta')} {DEG_DB} {DEG_OUT} {DEG_TMP} \
        --format-output 'query,target,pident,alnlen,qcov,tcov,evalue,bits' \
        -e 1e-5 -s 7.5 --max-seqs 1 --threads 4 -v 0
    deg_df   = pd.read_csv(DEG_OUT, sep='\t', names=['query','target','pident','alnlen','qcov','tcov','evalue','bits'])
    deg_filt = deg_df[(deg_df['qcov']>=QCOV_CUTOFF)&(deg_df['tcov']>=TCOV_CUTOFF)]
    def acc_from_header(h):
        parts = str(h).split('|')
        return parts[1] if len(parts) >= 2 else str(h).split()[0]
    essential_accs = set(deg_filt['query'].apply(acc_from_header).unique())
    deg_df.to_csv(RESULTS_DIR/'deg_hits_raw.tsv',       sep='\t', index=False)
    deg_filt.to_csv(RESULTS_DIR/'deg_hits_filtered.tsv', sep='\t', index=False)
    print(f'Essential candidates: {len(essential_accs)}')
    save_checkpoint({'essential_accs':list(essential_accs)})

classif_df = pd.read_csv(RESULTS_DIR/'strusel_classification.tsv', sep='\t')
classif_df['essential'] = classif_df['accession'].isin(essential_accs)
classif_df.to_csv(RESULTS_DIR/'strusel_classification.tsv', sep='\t', index=False)

priority   = classif_df[classif_df['essential']&classif_df['category'].isin(['D','D*'])]
mimic_risk = classif_df[classif_df['essential']&classif_df['category'].isin(['C','C2'])]
priority.to_csv(RESULTS_DIR/'strusel_priority_targets.tsv', sep='\t', index=False)
mimic_risk.to_csv(RESULTS_DIR/'strusel_mimic_risk.tsv',     sep='\t', index=False)
print(f'Priority targets (ess+D/D*): {len(priority)}')
print(f'Mimic risk       (ess+C/C2): {len(mimic_risk)}')
save_checkpoint({'priority_targets':list(priority['accession']),'mimic_risk':list(mimic_risk['accession'])})
print('Cell 9 done.')

In [ ]:
# Cell 10: CARD RGI AMR mapping + cross-reference
# Requires: diamond-aligner (installed in Cell 1)

import pandas as pd

if checkpoint_exists('amr_accs'):
    amr_accs = set(load_checkpoint()['amr_accs'])
    print(f'Checkpoint found. AMR-associated: {len(amr_accs)}')
else:
    print('Downloading CARD database...')
    !wget -q --show-progress -O /content/strusel/card.tar.bz2 https://card.mcmaster.ca/latest/data
    !mkdir -p /content/strusel/card
    !tar -xjf /content/strusel/card.tar.bz2 -C /content/strusel/card/
    !rgi load --card_json /content/strusel/card/card.json --local
    print('Running RGI (DIAMOND)...')
    !rgi main \
        --input_sequence {str(FASTA_DIR/'kpne.fasta')} \
        --output_file /content/strusel/results/rgi_output \
        --input_type protein --alignment_tool DIAMOND --local --clean -n 2
    rgi_df = pd.read_csv('/content/strusel/results/rgi_output.txt', sep='\t')
    def acc_from_rgi(orf_id):
        parts = str(orf_id).split('|')
        return parts[1] if len(parts) >= 2 else str(orf_id).split()[0]
    rgi_df['accession'] = rgi_df['ORF_ID'].apply(acc_from_rgi)
    amr_accs = set(rgi_df['accession'].unique())
    rgi_df.to_csv(RESULTS_DIR/'rgi_output_parsed.tsv', sep='\t', index=False)
    print(f'AMR-associated proteins: {len(amr_accs)}')
    print(rgi_df['AMR Gene Family'].value_counts().head(10))
    save_checkpoint({'amr_accs':list(amr_accs)})

classif_df = pd.read_csv(RESULTS_DIR/'strusel_classification.tsv', sep='\t')
classif_df['amr_associated'] = classif_df['accession'].isin(amr_accs)
classif_df.to_csv(RESULTS_DIR/'strusel_classification.tsv', sep='\t', index=False)

top_amr = classif_df[classif_df['amr_associated']&classif_df['essential']&classif_df['category'].isin(['D','D*'])]
top_amr.to_csv(RESULTS_DIR/'strusel_top_amr_targets.tsv', sep='\t', index=False)
print(f'Top AMR targets (ess+D/D*+AMR): {len(top_amr)}')
save_checkpoint({'amr_accs':list(amr_accs),'top_amr_targets':list(top_amr['accession'])})
print('Cell 10 done.')

In [ ]:
# Cell 11: Revised target tiers + PDF report
# Tier 1: essential + D/D* + NOT AMR (prime targets)
# Tier 2: essential + D/D* + AMR (resistance-prone)
# Tier 3: essential + C/C2 + NOT AMR (structural mimics)
# Tier 4: essential + C/C2 + AMR (highest risk)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

classif_df = pd.read_csv(RESULTS_DIR/'strusel_classification.tsv', sep='\t')
plddt_df   = pd.read_csv(RESULTS_DIR/'kpne_plddt_filter.tsv',      sep='\t')
rgi_df     = pd.read_csv(RESULTS_DIR/'rgi_output_parsed.tsv',       sep='\t')

tier1 = classif_df[classif_df['essential']&classif_df['category'].isin(['D','D*'])&~classif_df['amr_associated']]
tier2 = classif_df[classif_df['essential']&classif_df['category'].isin(['D','D*'])& classif_df['amr_associated']]
tier3 = classif_df[classif_df['essential']&classif_df['category'].isin(['C','C2'])&~classif_df['amr_associated']]
tier4 = classif_df[classif_df['essential']&classif_df['category'].isin(['C','C2'])& classif_df['amr_associated']]

tier1.to_csv(RESULTS_DIR/'strusel_tier1_prime_targets.tsv', sep='\t', index=False)
tier2.to_csv(RESULTS_DIR/'strusel_tier2_amr_targets.tsv',   sep='\t', index=False)
tier3.to_csv(RESULTS_DIR/'strusel_tier3_mimic_risk.tsv',    sep='\t', index=False)
tier4.to_csv(RESULTS_DIR/'strusel_tier4_amr_mimic.tsv',     sep='\t', index=False)

print('='*55)
print('  StruSel — Revised Target Tiers')
print('='*55)
print(f'  Tier 1 | Prime targets (ess+D/D*, no AMR)  : {len(tier1)}')
print(f'  Tier 2 | AMR-assoc    (ess+D/D*, AMR)      : {len(tier2)}')
print(f'  Tier 3 | Mimic risk   (ess+C/C2, no AMR)   : {len(tier3)}')
print(f'  Tier 4 | AMR+mimic    (ess+C/C2, AMR)      : {len(tier4)}')
print('='*55)

COLORS = {'A':'#4e79a7','B':'#f28e2b','B2':'#ffbe7d','C':'#e15759','C2':'#ff9d9a','D':'#59a14f','D*':'#8cd17d'}
cat_labels = {'A':'Homolog','B':'Seq-sim Struct-div','B2':'Seq-sim Struct-partial',
              'C':'Structural mimic','C2':'Partial mimic','D':'StruSel candidate','D*':'High-conf StruSel'}
cat_counts = classif_df['category'].value_counts()
cats = [c for c in ['A','B','B2','C','C2','D','D*'] if c in cat_counts]

pdf_path = str(RESULTS_DIR/'StruSel_Report_Klebsiella.pdf')
with PdfPages(pdf_path) as pdf:
    # Page 1: Title
    fig, ax = plt.subplots(figsize=(11,8.5)); ax.axis('off')
    ax.text(0.5,0.92,'StruSel',fontsize=36,fontweight='bold',ha='center',transform=ax.transAxes,color='#2d6a4f')
    ax.text(0.5,0.84,'Structurome-wide Selectivity of pathogen-exclusive targets',fontsize=14,ha='center',transform=ax.transAxes)
    ax.text(0.5,0.77,'Klebsiella pneumoniae vs Human Structurome',fontsize=13,ha='center',transform=ax.transAxes,fontstyle='italic')
    summary = f'Pathogen: {PATHOGEN_NAME} (TaxID: {PATHOGEN_TAXID})\nTotal proteome: {len(plddt_df)}\npLDDT-filtered (>=70): {plddt_df["kept"].sum()}\n\n── Key Results ──\nTier 1 Prime targets: {len(tier1)}\nTier 2 AMR-associated: {len(tier2)}\nTier 3 Structural mimics: {len(tier3)}\nTier 4 AMR+mimic: {len(tier4)}'
    ax.text(0.15,0.62,summary,fontsize=11,va='top',transform=ax.transAxes,family='monospace',bbox=dict(boxstyle='round,pad=0.5',facecolor='#f0f7f4',edgecolor='#2d6a4f',alpha=0.8))
    ax.text(0.5,0.04,'Pranavathiyani G | SASTRA Deemed University | Bio-GRID AMR Lab',fontsize=9,ha='center',color='#888',transform=ax.transAxes)
    pdf.savefig(fig,bbox_inches='tight'); plt.close()
    # Page 2: Category distribution
    fig, axes = plt.subplots(1,2,figsize=(11,6))
    axes[0].pie([cat_counts[c] for c in cats],labels=[f'{c}\n({cat_counts[c]})' for c in cats],colors=[COLORS[c] for c in cats],autopct='%1.1f%%',startangle=140,textprops={'fontsize':8})
    axes[0].set_title('Category Distribution',fontweight='bold')
    ess_counts = classif_df[classif_df['essential']]['category'].value_counts()
    x = np.arange(len(cats))
    axes[1].bar(x,[cat_counts.get(c,0) for c in cats],color=[COLORS[c] for c in cats],alpha=0.5,label='All')
    axes[1].bar(x,[ess_counts.get(c,0) for c in cats],color=[COLORS[c] for c in cats],alpha=1.0,label='Essential',edgecolor='black')
    axes[1].set_xticks(x); axes[1].set_xticklabels(cats); axes[1].legend()
    axes[1].set_title('All vs Essential by category',fontweight='bold')
    plt.tight_layout(); pdf.savefig(fig,bbox_inches='tight'); plt.close()
    # Page 3: pLDDT + TM-score
    fig, axes = plt.subplots(1,2,figsize=(11,5))
    axes[0].hist(plddt_df[plddt_df['kept']]['mean_plddt'],bins=30,color='#2d6a4f',alpha=0.7,label=f'Kept ({plddt_df["kept"].sum()})')
    axes[0].hist(plddt_df[~plddt_df['kept']]['mean_plddt'],bins=30,color='#e63946',alpha=0.7,label=f'Rejected ({(~plddt_df["kept"]).sum()})')
    axes[0].axvline(70,color='black',linestyle='--',label='Cutoff=70'); axes[0].legend()
    axes[0].set_title('pLDDT distribution',fontweight='bold')
    fs_best = pd.read_csv(RESULTS_DIR/'foldseek_best_hits.tsv',sep='\t')
    axes[1].hist(fs_best['qtmscore'],bins=30,color='#4e79a7',alpha=0.8)
    axes[1].axvline(0.5,color='red',linestyle='--',label='Homolog 0.5')
    axes[1].axvline(0.3,color='orange',linestyle='--',label='Unrelated 0.3'); axes[1].legend()
    axes[1].set_title('FoldSeek TM-score distribution',fontweight='bold')
    plt.tight_layout(); pdf.savefig(fig,bbox_inches='tight'); plt.close()
    # Page 4: Tier bar
    fig, ax = plt.subplots(figsize=(8,5))
    tiers=['Tier 1\nPrime','Tier 2\nAMR','Tier 3\nMimic','Tier 4\nAMR+Mimic']
    counts=[len(tier1),len(tier2),len(tier3),len(tier4)]
    bars=ax.bar(tiers,counts,color=['#2d6a4f','#e76f51','#e9c46a','#e63946'],edgecolor='black',alpha=0.85)
    for bar,cnt in zip(bars,counts): ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+10,str(cnt),ha='center',fontsize=11,fontweight='bold')
    ax.set_ylabel('Number of proteins'); ax.set_title('StruSel Target Tiers',fontweight='bold',fontsize=13)
    plt.tight_layout(); pdf.savefig(fig,bbox_inches='tight'); plt.close()

print(f'PDF saved: {pdf_path}')
save_checkpoint({'tier1_prime':list(tier1['accession']),'tier2_amr':list(tier2['accession']),'tier3_mimic':list(tier3['accession']),'tier4_amr_mimic':list(tier4['accession'])})
print('Cell 11 done.')

In [ ]:
# Cell 12: UniProt GO/pathway annotation for Tier 1 targets

import requests, re, time
import pandas as pd
from tqdm import tqdm
from collections import Counter

tier1_accs = load_checkpoint()['tier1_prime']
print(f'Tier 1 targets to annotate: {len(tier1_accs)}')

def fetch_uniprot_batch(accessions, batch_size=100):
    all_records = []
    fields = 'accession,protein_name,gene_names,reviewed,length,go,cc_subcellular_location,keyword,ec'
    for i in tqdm(range(0, len(accessions), batch_size)):
        batch = accessions[i:i+batch_size]
        query = ' OR '.join([f'accession:{acc}' for acc in batch])
        url   = f'https://rest.uniprot.org/uniprotkb/search?query={query}&format=tsv&fields={fields}&size={batch_size}'
        for attempt in range(3):
            try:
                r = requests.get(url, timeout=60)
                if r.status_code == 200:
                    lines = r.text.strip().split('\n')
                    if len(lines) > 1:
                        for line in lines[1:]:
                            if line.strip(): all_records.append(line.split('\t'))
                    break
                time.sleep(2)
            except: time.sleep(2)
    cols = ['accession','protein_name','gene_names','reviewed','length','go','subcellular_location','keywords','ec']
    padded = [r+['']*(len(cols)-len(r)) for r in all_records]
    return pd.DataFrame(padded, columns=cols)

annot_df = fetch_uniprot_batch(tier1_accs, batch_size=100)
annot_df.to_csv(RESULTS_DIR/'tier1_uniprot_annotations.tsv', sep='\t', index=False)
print(f'Annotated: {len(annot_df)} / {len(tier1_accs)}')

print('\n── Reviewed vs Unreviewed ──────────────────────────────')
print(annot_df['reviewed'].value_counts().to_string())

print('\n── Subcellular Localization (top 15) ───────────────────')
loc_counts = Counter()
for loc in annot_df['subcellular_location'].dropna():
    for m in re.findall(r'(?:Cell|Cytoplasm|Periplasm|Membrane|Nuclear)[^{;.]*', loc):
        loc_counts[m.strip()[:50]] += 1
for loc,cnt in loc_counts.most_common(15): print(f'  {cnt:4d}  {loc}')

bp_kw = ['process','biosynthesis','metabolism','regulation','replication','synthesis','transport','assembly','division','cycle','response']
mf_kw = ['activity','binding','catalytic','transferase','kinase','hydrolase','ligase','isomerase','lyase']
go_bp = Counter(); go_mf = Counter(); go_cc = Counter()
for go_str in annot_df['go'].dropna():
    for entry in go_str.split(';'):
        name = entry.split('[')[0].strip().lower()
        if not name: continue
        if   any(k in name for k in bp_kw): go_bp[name[:60]] += 1
        elif any(k in name for k in mf_kw): go_mf[name[:60]] += 1
        else:                                go_cc[name[:60]] += 1

print('\n── GO Biological Process (top 20) ──────────────────────')
for t,c in go_bp.most_common(20): print(f'  {c:4d}  {t}')

print('\n── GO Molecular Function (top 15) ──────────────────────')
for t,c in go_mf.most_common(15): print(f'  {c:4d}  {t}')

print('\n── GO Cellular Component (top 10) ──────────────────────')
for t,c in go_cc.most_common(10): print(f'  {c:4d}  {t}')

print('\n── Keywords (top 20) ───────────────────────────────────')
kw_counts = Counter()
for kw in annot_df['keywords'].dropna():
    for k in kw.split(';'):
        k = k.strip()
        if k: kw_counts[k] += 1
for kw,cnt in kw_counts.most_common(20): print(f'  {cnt:4d}  {kw}')

ec_count = annot_df['ec'].apply(lambda x: x != '').sum()
print(f'\n── Enzymatic targets: {ec_count} / {len(annot_df)}')
print('Cell 12 done.')

---
## TODO — Next Iterations

- [ ] ESM2 full human proteome comparison (better batching strategy)
- [ ] Pan-ESKAPE extension (change PATHOGEN_TAXID + AF URL in Cell 2)
- [ ] Druggability filter (fpocket / P2Rank on Tier 1 structures)
- [ ] Functional annotation enrichment plots
- [ ] GitHub Pages interactive summary
- [ ] Manuscript figures

---
*StruSel v0.1-alpha | Pranavathiyani G | SASTRA Deemed University *